# ETL pipelines that survive Tuesday — the hands-on half

The practical companion to **`etl_slides.html`**. The point of this session is not "read a CSV,
write a table". It is the four things that decide whether your pipeline still works in six
months: **idempotency, incrementality, validation, and recovery**.

You write three functions. **Airflow** does the rest — it decides which day to run, retries the
task that failed, and remembers what happened. That split is the whole lesson.

| Part | Deck slides | What you do |
|---|---|---|
| 0 · Setup | 12 | install, sandbox, a source that behaves like a real one |
| 1 · E, T, L | 5 | extract → transform → load into SQLite with SQLAlchemy |
| 2 · Idempotency | 16 | run it twice; prove the table does not double |
| 3 · The DAG | 17–18 | hand the job to Airflow; the logical date replaces the watermark |
| 4 · Validation | 19–20 | reject a bad batch **before** it reaches the table |
| 5 · Retries and history | 21–22 | `retries=2` instead of a retry loop; task history instead of a log table |

**The problem this solves:** the first version of every pipeline works. It is the second run,
the one with duplicate rows, a changed schema and a half-finished load, that teaches you what
the job actually was.

Everything runs in a throwaway `etl_demo/` folder; the last cell deletes it, Airflow included.

<!-- ar -->
<div dir="rtl" lang="ar">

**خطوط ETL تصمد يوم الثلاثاء: الجزء العملي**

هذا الدفتر هو الجانب العملي لعرض `etl_slides.html`. هدف الجلسة ليس "اقرأ ملف CSV واكتب جدولاً". الهدف أربعة أشياء تقرر هل سيبقى خطك يعمل بعد ستة أشهر: **عدم التكرار عند إعادة التشغيل (idempotency)، والتحميل التدريجي، والتحقق، والتعافي من الأعطال**.

أنت تكتب ثلاث دوال، و**Airflow** يتكفّل بالباقي: يقرر أي يوم يشغّل، ويعيد محاولة المهمة التي فشلت، ويتذكر ما جرى. هذا الفصل بين "المهمة" و"من يشغّلها" هو الدرس كله.

| الجزء | شرائح العرض | ماذا تفعل |
|---|---|---|
| ٠ · الإعداد | 12 | التثبيت، ومجلد التجربة، ومصدر بيانات يتصرف كالمصدر الحقيقي |
| ١ · E و T و L | 5 | استخراج ثم تحويل ثم تحميل إلى SQLite باستخدام SQLAlchemy |
| ٢ · عدم التكرار | 16 | شغّله مرتين، وأثبت أن الجدول لا يتضاعف |
| ٣ · الـ DAG | 17–18 | سلّم المهمة إلى Airflow، والتاريخ المنطقي يحل محل العلامة المائية |
| ٤ · التحقق | 19–20 | رفض الدفعة السيئة **قبل** أن تصل إلى الجدول |
| ٥ · إعادة المحاولة والسجل | 21–22 | `retries=2` بدل حلقة إعادة المحاولة، وسجل المهام بدل جدول السجل |

**المشكلة التي يحلها هذا:** النسخة الأولى من أي خط بيانات تعمل. التشغيل الثاني، الذي فيه صفوف مكررة، ومخطط تغيّر، وتحميل لم يكتمل، هو الذي يعلّمك ما كانت المهمة فعلاً.

كل شيء يعمل داخل مجلد مؤقت اسمه `etl_demo/`، وآخر خلية تحذفه ومعه Airflow.

</div>


## Step 0.1 · Install

SQLAlchemy plus pandas, then **Airflow 3**. SQLite ships with Python, so there is still no
database server to run — and Airflow keeps its own bookkeeping in SQLite too.

Airflow is a big install (a few hundred MB, a minute or two). It is the one place this session
got heavier; Parts 3 and 5 are where you get it back. Always install it with the official
**constraints** file for your Python version — without it, pip picks incompatible versions.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٠٫١ · التثبيت**

SQLAlchemy مع pandas، ثم **Airflow 3**. و SQLite تأتي مع Python، فلا توجد قاعدة بيانات تحتاج تشغيلها، و Airflow أيضاً يحفظ سجلاته في SQLite.

Airflow تثبيت كبير (بضع مئات من الميغابايت، ودقيقة أو دقيقتان). هذا هو المكان الوحيد الذي صارت فيه الجلسة أثقل، والجزءان الثالث والخامس هما حيث تسترد هذا الثمن. ثبّته دائماً مع ملف **القيود** (constraints) الرسمي الموافق لنسخة Python عندك، فبدونه يختار pip نسخاً غير متوافقة.

</div>


In [1]:
import sys

PY = f"{sys.version_info.major}.{sys.version_info.minor}"          # 3.11, 3.12, ...
AIRFLOW = "3.3.2"
CONSTRAINTS = f"https://raw.githubusercontent.com/apache/airflow/constraints-{AIRFLOW}/constraints-{PY}.txt"

!pip install -q sqlalchemy pandas numpy matplotlib
!pip install -q "apache-airflow=={AIRFLOW}" --constraint "{CONSTRAINTS}"

# Housekeeping, not part of the lesson: pandas prints "Pandas requires version ... of
# numexpr / bottleneck" when those are a version behind. That is about this machine, and
# joblib reprints it from every worker process, which buries the results.
import os, warnings
warnings.filterwarnings("ignore", message="Pandas requires version")
os.environ["PYTHONWARNINGS"] = "ignore:Pandas requires version"   # workers inherit this


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
uni2ts 2.0.0 requires numpy~=1.26.0, but you have numpy 2.5.3 which is incompatible.
uni2ts 2.0.0 requires python-dotenv==1.0.0, but you have python-dotenv 1.2.3 which is incompatible.
uni2ts 2.0.0 requires scipy~=1.11.3, but you have scipy 1.16.3 which is incompatible.
uni2ts 2.0.0 requires torch<2.5,>=2.1, but you have torch 2.9.1 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.5.3 which is incompatible.
ibm-cos-sdk-core 2.13.6 requires requests<2.32.3,>=2.32.0, but you have requests 2.34.2 which is incompatible.
autogluon-timeseries 1.5.0 requires gluonts<0.17,>=0.15.0, but you have gluonts 0.14.4 which is incompatible.
autogluon-timeseries 1.5.0 requires

<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تثبيت المكتبات**

بنبني رابط ملف القيود حسب نسخة Python يلي شغالة عندك، وبنثبت SQLAlchemy و pandas و numpy و matplotlib، وبعدين Airflow 3 مع ملف القيود. الأسطر الأخيرة بس بتسكّت تحذير pandas عن نسخ مكتبات مساعدة عشان ما يغطي على النتائج.

</div>


Now the tools. **SQLAlchemy** talks to the database for you: you describe tables in Python and
it writes the SQL. The database here is **SQLite** — a whole database in a single file, so
nothing needs installing or starting. **Airflow** is the orchestrator: it does not move your
data, it decides when your code runs and what happens when it fails.

<!-- ar -->
<div dir="rtl" lang="ar">

الآن الأدوات. **SQLAlchemy** تتعامل مع قاعدة البيانات نيابة عنك: تصف الجداول بلغة Python وهي تكتب SQL. قاعدة البيانات هنا **SQLite**: قاعدة بيانات كاملة في ملف واحد، فلا شيء يحتاج تثبيتاً أو تشغيلاً. و**Airflow** هو المنسّق: لا ينقل بياناتك، بل يقرر متى يعمل كودك وماذا يحدث حين يفشل.

</div>


In [2]:
%matplotlib inline
import sqlalchemy, pandas as pd, numpy as np
print("sqlalchemy", sqlalchemy.__version__)
print("pandas    ", pd.__version__)

sqlalchemy 2.0.52
pandas     2.3.3


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · التحقق من النسخ**

بنفعّل `%matplotlib inline` للرسومات، وبنستورد المكتبات، وبنطبع نسخة SQLAlchemy و pandas عشان نعرف على أي نسخة شغالين.

</div>

## Step 0.2 · A sandbox to work in

Everything — the CSVs, the warehouse, the DAG files and Airflow's own database — goes in one
throwaway folder. `AIRFLOW_HOME` is what makes that possible: point it inside the sandbox and
Airflow leaves nothing on your machine.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٠٫٢ · مجلد للتجربة**

كل شيء (ملفات CSV، والمستودع، وملفات الـ DAG، وقاعدة بيانات Airflow نفسها) داخل مجلد واحد مؤقت. المتغير `AIRFLOW_HOME` هو ما يجعل ذلك ممكناً: وجّهه داخل المجلد المؤقت فلا يترك Airflow أي أثر على جهازك.

</div>


In [3]:
import os, shutil, pathlib

BASE = pathlib.Path.cwd()          # the folder this notebook lives in
PROJ = BASE / "etl_demo"           # a throwaway sandbox, deleted by the last cell

if PROJ.exists():
    shutil.rmtree(PROJ)            # re-running this notebook is always safe
(PROJ / "dags").mkdir(parents=True)
os.chdir(PROJ)

# Airflow keeps its database, config and logs in AIRFLOW_HOME. Inside the sandbox means
# the last cell deletes Airflow too. The other three are for readability, not correctness.
os.environ["AIRFLOW_HOME"] = str(PROJ / "airflow")
os.environ["AIRFLOW__CORE__DAGS_FOLDER"] = str(PROJ / "dags")
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"     # or 50 tutorial DAGs join yours
os.environ["AIRFLOW__LOGGING__LOGGING_LEVEL"] = "ERROR"  # Airflow logs a *lot* at INFO

print("working inside:", os.getcwd())
!airflow version

working inside: /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/05-etl/etl_demo
3.3.2


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مجلد التجربة و AIRFLOW_HOME**

بننشئ مجلد `etl_demo` وجواته مجلد `dags`، وبنحذفه أول إذا كان موجود عشان إعادة تشغيل الدفتر تضل آمنة. بعدين بنظبط `AIRFLOW_HOME` جوا المجلد المؤقت، و `DAGS_FOLDER` عالمجلد يلي رح نكتب فيه الـ DAG. و `LOAD_EXAMPLES=False` عشان ما تطلع خمسين DAG جاهزة مع تبعوننا، و `LOGGING_LEVEL=ERROR` عشان الإخراج يضل مقروء. وآخر سطر `airflow version` بيتأكد إنه Airflow متثبت فعلاً.

</div>


## Step 0.3 · A source that behaves like a real one

Real sources are not clean. This generator gives us, on purpose:

* rows that arrive **late** (an event from yesterday shows up today)
* **duplicate** event ids, because the upstream system retries
* **missing** values and a few impossible ones
* a timestamp we can use as a **watermark**

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٠٫٣ · مصدر يتصرف كالمصدر الحقيقي**

المصادر الحقيقية ليست نظيفة. هذا المولّد يعطينا عن قصد:

* صفوفاً تصل **متأخرة** (حدث من الأمس يظهر اليوم)
* أرقام أحداث **مكررة**، لأن النظام المصدر يعيد المحاولة
* قيماً **مفقودة** وبعض القيم المستحيلة
* وقتاً نستطيع استخدامه **علامة مائية**

</div>

In [4]:
import os
from datetime import datetime, timedelta

rng = np.random.default_rng(7)
START = datetime(2026, 3, 1)

def make_batch(day: int, n: int = 500, dupes: int = 12, late: int = 20) -> pd.DataFrame:
    """One day's export from an upstream system, warts and all."""
    base = START + timedelta(days=day)
    ids = np.arange(day * 1000, day * 1000 + n)
    df = pd.DataFrame({
        "event_id":    ids,
        "customer_id": rng.integers(1, 900, n),
        "amount":      rng.normal(60, 25, n).round(2),
        "currency":    rng.choice(["USD", "EUR", "usd"], n, p=[.7, .2, .1]),   # messy case
        "occurred_at": [base + timedelta(minutes=int(m)) for m in rng.integers(0, 1440, n)],
    })
    df.loc[rng.random(n) < .04, "amount"] = np.nan          # holes
    df.loc[rng.random(n) < .01, "amount"] = -1.0            # impossible
    if dupes:                                               # upstream retried
        df = pd.concat([df, df.sample(dupes, random_state=day)], ignore_index=True)
    if late:                                                # yesterday, arriving today
        l = make_batch(day - 1, n=late, dupes=0, late=0) if day > 0 else None
        if l is not None: df = pd.concat([df, l], ignore_index=True)
    return df.sample(frac=1, random_state=day).reset_index(drop=True)

os.makedirs("landing", exist_ok=True)
for day in range(3):
    b = make_batch(day)
    b.to_csv(f"landing/events_day{day}.csv", index=False)
    print(f"day {day}: {len(b):4} rows, {b.event_id.duplicated().sum():3} duplicate ids, "
          f"{b.amount.isna().sum():2} missing amounts")

day 0:  512 rows,  12 duplicate ids, 22 missing amounts
day 1:  532 rows,  12 duplicate ids, 28 missing amounts
day 2:  532 rows,  12 duplicate ids, 17 missing amounts


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مصدر بيانات فوضوي زي الحقيقي**

بنكتب دالة `make_batch` بتولّد تصدير يوم واحد فيه كل مشاكل الواقع: عملة مكتوبة بأحرف صغيرة أحياناً، وقيم مفقودة، ومبالغ سالبة مستحيلة، وصفوف مكررة كأنه المصدر أعاد المحاولة، وصفوف من اليوم السابق وصلت متأخرة. بعدين بنحفظ ثلاث أيام كملفات CSV بمجلد `landing` وبنطبع ملخص كل يوم.

</div>

---
# Part 1 — Extract, transform, load   ·   deck slide 5

Three functions, kept deliberately separate. That separation is not tidiness: it is what lets
you test the transform without a database, re-load without re-extracting, and — in Part 3 —
hand each one to Airflow as its own task.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الأول: الاستخراج، والتحويل، والتحميل**

ثلاث دوال، منفصلة عن قصد. هذا الفصل ليس للترتيب فقط: هو الذي يسمح لك باختبار التحويل دون قاعدة بيانات، وإعادة التحميل دون إعادة الاستخراج، و(في الجزء الثالث) تسليم كل دالة إلى Airflow كمهمة مستقلة.

</div>


In [ ]:
%%writefile etl.py
"""A small ETL job, written so that each stage can be tested on its own."""
from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

log = logging.getLogger("etl")

HERE = Path(__file__).resolve().parent      # so the job works from anywhere -- Airflow included

SCHEMA = {"event_id": "int64", "customer_id": "int64", "amount": "float64",
          "currency": "object", "occurred_at": "datetime64[ns]"}


def get_engine():
    """One place that knows where the warehouse lives."""
    return create_engine(f"sqlite:///{HERE / 'warehouse.db'}", future=True)

# ---------------------------------------------------------------- E
def extract(path: str | Path) -> pd.DataFrame:
    """Read the source. Nothing clever -- and nothing destructive."""
    df = pd.read_csv(path, parse_dates=["occurred_at"])
    log.info("extracted %d rows from %s", len(df), path)
    return df

# ---------------------------------------------------------------- T
def transform(df: pd.DataFrame) -> pd.DataFrame:
    """Pure function: same input, same output, no database, no clock, no network.

    That purity is why this is the only stage with real business logic -- it is the
    only one you can unit-test in milliseconds.
    """
    out = df.copy()
    out["currency"] = out.currency.str.upper().str.strip()          # usd -> USD
    out = out[out.amount.notna() & (out.amount >= 0)]               # drop impossible rows
    out["amount_eur"] = out.amount * out.currency.map({"USD": 0.92, "EUR": 1.0}).fillna(1.0)
    out["occurred_date"] = out.occurred_at.dt.date.astype("string")
    out = out.drop_duplicates(subset=["event_id"], keep="last")      # upstream retries
    return out[["event_id", "customer_id", "amount", "currency",
                "amount_eur", "occurred_at", "occurred_date"]]

# ---------------------------------------------------------------- L
def load(df: pd.DataFrame, engine, table: str = "events") -> int:
    """Append rows. Part 2 replaces this with something safe to run twice."""
    df.to_sql(table, engine, if_exists="append", index=False)
    log.info("loaded %d rows into %s", len(df), table)
    return len(df)

Overwriting etl.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملف ETL بثلاث دوال**

`%%writefile` بتكتب `etl.py`. فيه `HERE` بتحدد مكان الملف عشان الشغل يزبط حتى لما Airflow يشغّله من مجلد تاني، و `get_engine` بترجع الاتصال بقاعدة البيانات من مكان واحد. و `extract` بتقرأ الملف بس، و `transform` دالة نقية: بتوحّد العملة، وبتشيل المبالغ المفقودة والسالبة، وبتحسب المبلغ باليورو، وبتشيل التكرار حسب `event_id`. و `load` بتضيف الصفوف للجدول بـ `to_sql`. فصلناهم عشان نقدر نختبر التحويل بدون قاعدة بيانات.

</div>


`etl.py` is now a real module, so `import etl` gives you `extract`, `transform` and `load` as
ordinary functions. `importlib.reload` picks up edits without restarting the kernel.

<!-- ar -->
<div dir="rtl" lang="ar">

`etl.py` الآن وحدة حقيقية، فـ `import etl` يعطيك `extract` و `transform` و `load` كدوال عادية. و `importlib.reload` يلتقط التعديلات دون إعادة تشغيل النواة.

</div>

In [7]:
import logging, importlib, etl
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
importlib.reload(etl)

from sqlalchemy import create_engine, text
engine = create_engine("sqlite:///warehouse.db", future=True)

raw = etl.extract("landing/events_day0.csv")
clean = etl.transform(raw)
print(f"\n{len(raw)} rows in -> {len(clean)} rows out")
print(f"  dropped {len(raw) - len(clean)} (duplicates, missing and negative amounts)")
clean.head(3)

INFO extracted 512 rows from landing/events_day0.csv



512 rows in -> 471 rows out
  dropped 41 (duplicates, missing and negative amounts)


,event_id,customer_id,amount,currency,amount_eur,occurred_at,occurred_date
1,100,617,96.98,EUR,96.9800,2026-03-01 14:08:00,2026-03-01
3,46,104,82.92,USD,76.2864,2026-03-01 10:27:00,2026-03-01
4,374,391,67.81,USD,62.3852,2026-03-01 14:03:00,2026-03-01


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · استخراج وتحويل**

بنفعّل السجلات، وبنعيد تحميل `etl` بـ `importlib.reload` عشان أي تعديل ينقرأ. بنفتح قاعدة SQLite بـ `create_engine`، وبعدين بنستخرج ملف اليوم الأول وبنحوله، وبنطبع كم صف دخل وكم طلع وكم انشال.

</div>

Load it, then count the rows. Remember the number — the next cell runs the *same* file again,
and the count is the whole point.

<!-- ar -->
<div dir="rtl" lang="ar">

حمّل البيانات، ثم عُدّ الصفوف. تذكّر الرقم، فالخلية التالية تشغّل *نفس* الملف مرة أخرى، والعدد هو المقصود كله.

</div>

In [8]:
etl.load(clean, engine)
with engine.connect() as c:
    print("rows in events:", c.execute(text("select count(*) from events")).scalar())

INFO loaded 471 rows into events


rows in events: 471


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · التحميل وعد الصفوف**

بنحمّل الصفوف النظيفة لجدول `events`، وبعدين بنعدّ الصفوف بجملة SQL. تذكّر هاد الرقم، لأنه الخلية الجاية بتحمّل نفس الملف مرة تانية.

</div>

### Why the transform is a pure function

`transform` takes a DataFrame and returns a DataFrame. No database, no clock, no network. So:

* you can test it with three hand-written rows and no infrastructure;
* a failure is reproducible from the input alone;
* re-running it on the same input always gives the same answer, which is what Part 2 needs.

<!-- ar -->
<div dir="rtl" lang="ar">

**لماذا التحويل دالة نقية**

`transform` تأخذ DataFrame وترجع DataFrame. لا قاعدة بيانات، ولا ساعة، ولا شبكة. لذلك:

* تستطيع اختبارها بثلاثة صفوف مكتوبة يدوياً دون أي بنية تحتية؛
* أي فشل يمكن إعادة إنتاجه من المدخل وحده؛
* إعادة تشغيلها على نفس المدخل تعطي دائماً نفس الجواب، وهذا ما يحتاجه الجزء الثاني.

</div>

In [9]:
# the entire test suite for the business logic -- no database required
def test_currency_is_normalised():
    out = etl.transform(pd.DataFrame({"event_id":[1],"customer_id":[1],"amount":[10.0],
        "currency":[" usd "],"occurred_at":[pd.Timestamp("2026-03-01")]}))
    assert out.currency.iloc[0] == "USD"

def test_negative_amounts_are_dropped():
    out = etl.transform(pd.DataFrame({"event_id":[1,2],"customer_id":[1,1],"amount":[-5.0,5.0],
        "currency":["USD","USD"],"occurred_at":[pd.Timestamp("2026-03-01")]*2}))
    assert list(out.event_id) == [2]

def test_duplicate_ids_collapse():
    out = etl.transform(pd.DataFrame({"event_id":[1,1],"customer_id":[1,1],"amount":[5.0,7.0],
        "currency":["USD","USD"],"occurred_at":[pd.Timestamp("2026-03-01")]*2}))
    assert len(out) == 1 and out.amount.iloc[0] == 7.0      # keep="last"

for t in [test_currency_is_normalised, test_negative_amounts_are_dropped, test_duplicate_ids_collapse]:
    t(); print("PASS", t.__name__)

PASS test_currency_is_normalised
PASS test_negative_amounts_are_dropped
PASS test_duplicate_ids_collapse


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · اختبارات التحويل بدون قاعدة بيانات**

بنكتب ثلاث اختبارات صغيرة، كل واحد بيعطي `transform` جدول من صف أو صفين مكتوبين بإيدنا: العملة بتتوحد، والمبالغ السالبة بتنشال، والأرقام المكررة بتصير صف واحد. بنشغلهم وبنطبع PASS.

</div>

---
# Part 2 — Idempotency   ·   deck slide 16

**Idempotent** means: running it twice leaves the same result as running it once.

This is the single most valuable property an ETL job can have, because things *will* be re-run —
a retry, a backfill, someone clicking the button twice, an Airflow task that timed out after it
already succeeded.

Watch the naive `append` fail that test.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الثاني: عدم التكرار عند إعادة التشغيل**

**Idempotent** تعني: تشغيله مرتين يترك نفس النتيجة كتشغيله مرة واحدة.

هذه أثمن صفة يمكن أن تملكها مهمة ETL، لأن الأشياء *ستُعاد* حتماً: إعادة محاولة، أو إعادة معالجة بيانات قديمة، أو شخص ضغط الزر مرتين، أو مهمة Airflow انتهت مهلتها بعد أن نجحت فعلاً.

شاهد الإضافة البسيطة `append` تفشل في هذا الاختبار.

</div>


In [17]:
etl.load(clean, engine)                      # exactly the same batch, a second time
with engine.connect() as c:
    n = c.execute(text("select count(*) from events")).scalar()
    dupes = c.execute(text("""select count(*) from (
                                select event_id from events group by event_id having count(*) > 1)""")).scalar()
print(f"rows now: {n}   duplicated event_ids: {dupes}")
print("\nThe same file loaded twice doubled the table. This is the bug you find in a report.")

INFO loaded 471 rows into events


rows now: 3297   duplicated event_ids: 471

The same file loaded twice doubled the table. This is the bug you find in a report.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · التحميل مرتين: المشكلة**

بنحمّل نفس الدفعة مرة تانية بالضبط، وبنعدّ الصفوف وكم `event_id` مكرر. رح نشوف إنه الجدول تضاعف، وهاد الخطأ يلي بيطلع بعدين بالتقارير.

</div>

## Step 2.1 · A primary key and an upsert

Two changes make it idempotent:

1. a **primary key** on `event_id`, so the database itself refuses a duplicate;
2. an **upsert** — insert, or update the row that is already there.

SQLite and Postgres both spell it `ON CONFLICT`; SQLAlchemy gives you a dialect-specific helper.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٢٫١ · مفتاح أساسي و upsert**

تغييران يجعلانها لا تتكرر:

1. **مفتاح أساسي** على `event_id`، فترفض قاعدة البيانات نفسها أي تكرار؛
2. **upsert**: أدخل الصف، أو حدّث الصف الموجود مسبقاً.

SQLite و Postgres يكتبانه `ON CONFLICT`، و SQLAlchemy تعطيك أداة خاصة بكل نوع قاعدة بيانات.

</div>

In [11]:
%%writefile -a etl.py


# ---------------------------------------------------------------- L, done properly
from sqlalchemy import Column, DateTime, Float, Integer, MetaData, String, Table
from sqlalchemy.dialects.sqlite import insert as sqlite_insert

meta = MetaData()
events_v2 = Table("events_v2", meta,
    Column("event_id",      Integer, primary_key=True),          # <- the whole trick
    Column("customer_id",   Integer, nullable=False),
    Column("amount",        Float),
    Column("currency",      String(3)),
    Column("amount_eur",    Float),
    Column("occurred_at",   DateTime),
    Column("occurred_date", String(10)),
    Column("loaded_at",     DateTime),
)


def upsert(df: pd.DataFrame, engine=None) -> int:
    """Idempotent load: insert new rows, refresh rows we have seen before."""
    engine = engine or get_engine()
    meta.create_all(engine)
    if df.empty:
        return 0
    rows = df.assign(loaded_at=pd.Timestamp.utcnow().tz_localize(None)).to_dict("records")
    stmt = sqlite_insert(events_v2).values(rows)
    stmt = stmt.on_conflict_do_update(
        index_elements=["event_id"],
        set_={c: stmt.excluded[c] for c in df.columns if c != "event_id"})
    with engine.begin() as conn:                # one transaction: all of it, or none of it
        conn.execute(stmt)
    return len(rows)

Appending to etl.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مفتاح أساسي و upsert داخل الملف**

`%%writefile -a` بتضيف عالملف الموجود بدل ما تكتبه من جديد. بنعرّف جدول `events_v2` و `event_id` فيه مفتاح أساسي. دالة `upsert` بتستخدم `on_conflict_do_update`: لو الصف جديد بينضاف، ولو موجود بيتحدث، وكلها جوا `engine.begin()` يعني معاملة وحدة. حطيناها جوا `etl.py` مش بالدفتر عشان الـ DAG في الجزء الثالث يقدر يستوردها.

</div>


In [12]:
importlib.reload(etl)                        # pick up the appended code

print("first  load:", etl.upsert(clean, engine), "rows offered")
with engine.connect() as c:
    print("  table now:", c.execute(text("select count(*) from events_v2")).scalar())
print("second load:", etl.upsert(clean, engine), "rows offered")
with engine.connect() as c:
    print("  table now:", c.execute(text("select count(*) from events_v2")).scalar(),
          " <- unchanged. The job is idempotent.")

first  load: 471 rows offered
  table now: 471
second load: 471 rows offered
  table now: 471  <- unchanged. The job is idempotent.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · نفس الدفعة مرتين**

بنعمل `reload` للوحدة عشان الكود الجديد ينقرأ، وبعدين بنحمّل نفس الدفعة مرتين بـ `upsert`. العدد المعروض بيضل نفسه في المرتين، بس عدد صفوف الجدول ما بيتغير: هاي هي النتيجة المطلوبة.

</div>


Two counts, side by side: the append-only table against the upsert table, after loading the same
file twice. **Upsert** = update the row if its key is already there, insert it if not.

<!-- ar -->
<div dir="rtl" lang="ar">

رقمان جنباً إلى جنب: جدول الإضافة فقط مقابل جدول الـ upsert، بعد تحميل نفس الملف مرتين. **Upsert** تعني: حدّث الصف إذا كان مفتاحه موجوداً، وأدخله إذا لم يكن.

</div>

In [12]:
with engine.connect() as c:
    a = c.execute(text("select count(*) from events")).scalar()
    b = c.execute(text("select count(*) from events_v2")).scalar()
print(f"append-only table after two runs : {a}")
print(f"upsert table after two runs      : {b}")
assert b < a, "the upsert table must not have doubled"

append-only table after two runs : 942
upsert table after two runs      : 471


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مقارنة الجدولين**

بنعدّ الصفوف بالجدول القديم يلي بيضيف بس، وبالجدول الجديد يلي بيعمل upsert، بعد تحميل نفس الملف مرتين. الـ `assert` بتتأكد إنه الجدول الجديد ما تضاعف.

</div>

---
# Part 3 — The DAG   ·   deck slides 17–18

Your job works. Now something has to **run** it: every day, in order, retrying what fails and
remembering what happened. That something is an orchestrator, and the job is not it.

Here is the part worth slowing down for. A pipeline that processes "everything since last time"
needs to know when last time was. Hand-written, that is a watermark table: a row you read at the
start, a row you write at the end, and a long tail of bugs when the job dies between the two.

**Airflow already knows.** Every run has a `logical_date` — the day that run is *for*, decided by
the schedule and not by the clock. Ask for that date and the watermark table disappears, along
with the bugs. Re-run last Tuesday and you get last Tuesday's data, not today's.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الثالث: الـ DAG**

مهمتك تعمل. الآن يجب أن **يشغّلها** شيء ما: كل يوم، بالترتيب، يعيد محاولة ما يفشل ويتذكر ما جرى. هذا الشيء هو المنسّق، والمهمة ليست هي المنسّق.

هنا الجزء الذي يستحق التمهّل. الخط الذي يعالج "كل ما استجد منذ المرة الماضية" يحتاج أن يعرف متى كانت المرة الماضية. إذا كتبتها بيدك فهي جدول علامة مائية: صف تقرأه في البداية، وصف تكتبه في النهاية، وذيل طويل من الأخطاء حين تموت المهمة بينهما.

**و Airflow يعرف ذلك أصلاً.** لكل تشغيل تاريخ منطقي `logical_date`: اليوم الذي يخص هذا التشغيل، يقرره الجدول لا الساعة. اطلب هذا التاريخ فيختفي جدول العلامة المائية ومعه أخطاؤه. أعد تشغيل الثلاثاء الماضي فتحصل على بيانات الثلاثاء الماضي، لا بيانات اليوم.

</div>


In [13]:
%%writefile dags/etl_pipeline.py
"""The same three functions -- now with Airflow deciding when they run."""
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parent.parent))   # so `import etl` works

import pendulum
from airflow.sdk import dag, task            # Airflow 3's import path

import etl

START = pendulum.datetime(2026, 3, 1, tz="UTC")


@dag(schedule="@daily", start_date=START, catchup=False,
     default_args={"retries": 2, "retry_delay": pendulum.duration(seconds=2)})
def etl_pipeline():
    """One run per day. Airflow picks the day; no task ever calls now()."""

    @task
    def extract(**context):
        # the logical date IS the watermark -- no table, no bookkeeping, no drift
        day = (context["logical_date"] - START).days
        return str(etl.HERE / "landing" / f"events_day{day}.csv")

    @task
    def transform(path):
        clean = etl.transform(etl.extract(path))
        out = etl.HERE / "staging" / pathlib.Path(path).name
        out.parent.mkdir(exist_ok=True)
        clean.to_csv(out, index=False)
        print(f"{len(clean)} clean rows -> {out.name}")
        return str(out)                      # pass a path between tasks, never a DataFrame

    @task
    def load(path):
        import pandas as pd
        n = etl.upsert(pd.read_csv(path, parse_dates=["occurred_at"]))
        print(f"offered {n} rows to the warehouse")
        return n

    load(transform(extract()))                # this line is the graph


etl_pipeline()

Writing dags/etl_pipeline.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملف الـ DAG**

بنكتب الـ DAG بأسلوب TaskFlow: كل `@task` دالة عادية، و `@dag` بيوصفها كخط كامل. أول سطرين بيضيفوا مجلد المشروع لمسار الاستيراد عشان `import etl` تزبط لما Airflow يشغّل الملف. `schedule="@daily"` يعني تشغيل كل يوم، و `retries: 2` بتنطبق على كل المهام. لاحظ `context["logical_date"]`: هاد تاريخ التشغيل يلي Airflow قرره، وهو يلي بيحدد أي ملف ننقرا، بدل جدول علامة مائية. وبنمرر مسارات ملفات بين المهام مش DataFrame، لأن ما بينهم بيمر عبر قاعدة بيانات Airflow. وآخر سطر `load(transform(extract()))` هو الرسم البياني نفسه.

</div>


Airflow keeps its own bookkeeping in a SQLite file inside `AIRFLOW_HOME`, created once by
`db migrate`. Then run it — three days, one after another. `airflow dags test` executes a real DAG run in this
one process: no scheduler, no web server, no ports.

Watch the table grow. Each day brings some rows that belong to *yesterday* (late arrivals), and
the upsert absorbs them without duplicating anything.

<!-- ar -->
<div dir="rtl" lang="ar">

الآن شغّله: ثلاثة أيام، واحداً بعد الآخر. الأمر `airflow dags test` ينفّذ تشغيلاً حقيقياً داخل هذه العملية وحدها: بلا مجدول، وبلا خادم ويب، وبلا منافذ.

راقب الجدول وهو يكبر. كل يوم يجلب صفوفاً تخص *أمس* (وصلت متأخرة)، والـ upsert يستوعبها دون أي تكرار.

</div>


In [14]:
!airflow db migrate 2>&1 | tail -1
!airflow dags list 2>&1 | grep -E "dag_id|etl_pipeline"


def table_count():
    with engine.connect() as c:
        return c.execute(text("select count(*) from events_v2")).scalar()

for day in ("2026-03-01", "2026-03-02", "2026-03-03"):
    !airflow dags test etl_pipeline {day} 2>&1 | grep -E "clean rows|offered"
    print(f"   {day} -> table total: {table_count()}\n")

471 clean rows -> events_day0.csv
offered 471 rows to the warehouse


   2026-03-01 -> table total: 471



485 clean rows -> events_day1.csv
offered 485 rows to the warehouse


   2026-03-02 -> table total: 939



494 clean rows -> events_day2.csv
offered 494 rows to the warehouse


   2026-03-03 -> table total: 1415



<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إنشاء قاعدة Airflow وتشغيل ثلاثة أيام**

`airflow db migrate` بتنشئ جداول Airflow جوا `AIRFLOW_HOME` (بناخد آخر سطر بس لأن الإخراج طويل)، و `airflow dags list` بتتأكد إنه لقى الـ DAG بدون أخطاء. و `table_count` بترجع عدد صفوف الجدول. وبعدين لكل يوم من التواريخ التلاتة بنشغّل `airflow dags test` وبنفلتر الإخراج بـ `grep` عشان نشوف بس أسطرنا. لاحظ إنه إحنا ما مررنا أي علامة مائية: التاريخ يلي بنعطيه للأمر هو `logical_date`، والمهمة `extract` بتحسب منه أي ملف تقرا.

</div>


A retry is the same command again. Airflow re-runs the whole day; the upsert makes that free.

<!-- ar -->
<div dir="rtl" lang="ar">

إعادة المحاولة هي الأمر نفسه مرة أخرى. يعيد Airflow تشغيل اليوم كاملاً، والـ upsert يجعل ذلك بلا كلفة.

</div>


In [15]:
before = table_count()
!airflow dags test etl_pipeline 2026-03-03 2>&1 | grep -E "clean rows|offered"
after = table_count()

print(f"\ntable before retry {before}, after retry {after}")
assert before == after, "a retry must not change the table"
print("Idempotent + scheduled: Airflow may re-run any day it likes, and nothing breaks.")

494 clean rows -> events_day2.csv
offered 494 rows to the warehouse



table before retry 1415, after retry 1415
Idempotent + scheduled: Airflow may re-run any day it likes, and nothing breaks.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إعادة تشغيل نفس اليوم**

بنسجّل عدد الصفوف قبل، وبنشغّل نفس اليوم مرة تانية، وبنسجّل بعد. الـ `assert` بيفشل الخلية لو العدد تغيّر. هاي هي الصفة يلي بتخلي إعادة المعالجة والـ backfill آمنين.

</div>


### Why the logical date, and not `now()`

A task that calls `datetime.now()` gives a different answer every time it runs — so a re-run of
last Tuesday quietly processes *today*. That is the bug behind most "we backfilled and it made it
worse" stories.

`logical_date` is fixed for the run. Re-run it in a year and it still means that Tuesday. This is
the same discipline as the pure `transform` in Part 1: **no clock, no network, no hidden state**,
so the same input always gives the same output.

<!-- ar -->
<div dir="rtl" lang="ar">

**لماذا التاريخ المنطقي وليس `now()`؟**

المهمة التي تستدعي `datetime.now()` تعطي جواباً مختلفاً في كل تشغيل، فإعادة تشغيل الثلاثاء الماضي تعالج *اليوم* بهدوء. هذا هو الخطأ الكامن خلف معظم قصص "أعدنا المعالجة فازداد الأمر سوءاً".

أما `logical_date` فثابت لذلك التشغيل. أعد تشغيله بعد سنة فسيظل يعني ذلك الثلاثاء. هذا هو الانضباط نفسه الذي في دالة `transform` النقية في الجزء الأول: **لا ساعة، ولا شبكة، ولا حالة خفية**، فنفس المدخل يعطي دائماً نفس المخرج.

</div>


---
# Part 4 — Validate before you load   ·   deck slides 19–20

A pipeline that loads bad data is worse than one that fails, because the failure is loud and the
bad data is silent. Check at the boundary, and refuse the whole batch if it is wrong.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الرابع: تحقّق قبل أن تحمّل**

الخط الذي يحمّل بيانات سيئة أسوأ من خط يفشل، لأن الفشل صاخب والبيانات السيئة صامتة. افحص عند الحدود، وارفض الدفعة كاملة إذا كانت خاطئة.

</div>


In [16]:
%%writefile validate.py
"""Contract checks. Cheap, boring, and they save whole days."""
from __future__ import annotations
import pandas as pd

REQUIRED = ["event_id", "customer_id", "amount", "currency", "occurred_at"]


class DataContractError(Exception):
    """Raised instead of loading a batch we do not trust."""


def validate(df: pd.DataFrame, *, max_null_amount=0.10, currencies=("USD", "EUR")) -> dict:
    problems, stats = [], {"rows": len(df)}

    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        problems.append(f"missing columns: {missing}")           # schema drift

    if df.empty:
        problems.append("empty batch")

    if "event_id" in df:
        n_dupe = int(df.event_id.duplicated().sum())
        stats["duplicate_ids"] = n_dupe                          # tolerated: the upsert handles it

    if "amount" in df:
        null_rate = float(df.amount.isna().mean())
        stats["null_amount_rate"] = round(null_rate, 4)
        if null_rate > max_null_amount:                          # a spike means upstream broke
            problems.append(f"null amount rate {null_rate:.1%} over {max_null_amount:.0%}")

    if "currency" in df:
        unknown = sorted(set(df.currency.str.upper().str.strip()) - set(currencies))
        stats["unknown_currencies"] = unknown
        if unknown:
            problems.append(f"unknown currencies: {unknown}")

    if "occurred_at" in df:
        future = int((pd.to_datetime(df.occurred_at) > pd.Timestamp.utcnow().tz_localize(None)
                      + pd.Timedelta("1D")).sum())
        stats["future_rows"] = future
        if future:
            problems.append(f"{future} rows dated in the future")

    if problems:
        raise DataContractError("; ".join(problems))
    return stats

Writing validate.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملف التحقق**

`%%writefile` بتكتب `validate.py`. فيه استثناء خاص `DataContractError`، ودالة `validate` بتفحص: الأعمدة المطلوبة موجودة، والدفعة مش فاضية، ونسبة المبالغ الفاضية مش عالية، وما في عملات غريبة، وما في تواريخ بالمستقبل. وبتحسب كمان عدد التكرارات بس ما بترفض عليها لأنه الـ upsert بيعالجها. لو في أي مشكلة بترفع الاستثناء بكل الأسباب.

</div>

Validation now. `validate.py` holds the checks; a batch that fails one is refused **before** it
reaches the table, because a rejected batch is cheaper than a corrupted one.

<!-- ar -->
<div dir="rtl" lang="ar">

التحقق الآن. `validate.py` يحتوي الفحوصات، وأي دفعة تفشل في واحد منها تُرفض **قبل** أن تصل إلى الجدول، لأن الدفعة المرفوضة أرخص من الدفعة التي تفسد البيانات.

</div>

In [17]:
import importlib, validate
importlib.reload(validate)

good = etl.extract("landing/events_day0.csv")
print("clean batch ->", validate.validate(good))

INFO extracted 512 rows from landing/events_day0.csv


clean batch -> {'rows': 512, 'duplicate_ids': 12, 'null_amount_rate': 0.043, 'unknown_currencies': [], 'future_rows': 0}


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تجربة دفعة سليمة**

بنعيد تحميل `validate`، وبنستخرج ملف اليوم الأول، وبنفحصه. بيعدّي وبيرجع إحصاءات الدفعة.

</div>

Four broken batches, each broken in a different realistic way. None of them should get in.

<!-- ar -->
<div dir="rtl" lang="ar">

أربع دفعات مكسورة، كل واحدة مكسورة بطريقة واقعية مختلفة. لا يجب أن تدخل أي واحدة منها.

</div>

In [18]:
# now four batches that a real upstream will eventually send you
broken = {
    "a column disappeared":   good.drop(columns=["amount"]),
    "half the amounts null":  good.assign(amount=lambda d: d.amount.where(np.arange(len(d)) % 2 == 0)),
    "a new currency":         good.assign(currency=lambda d: d.currency.mask(np.arange(len(d)) < 5, "GBP")),
    "timestamps in the future": good.assign(occurred_at=lambda d: d.occurred_at + pd.Timedelta("400D")),
}
for name, df in broken.items():
    try:
        validate.validate(df)
        print(f"  {name:26} -> ACCEPTED (should not happen)")
    except validate.DataContractError as e:
        print(f"  {name:26} -> refused: {e}"[:112])

  a column disappeared       -> refused: missing columns: ['amount']
  half the amounts null      -> refused: null amount rate 52.0% over 10%
  a new currency             -> refused: unknown currencies: ['GBP']
  timestamps in the future   -> refused: 512 rows dated in the future


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · أربع دفعات مكسورة**

بنعمل أربع نسخ مكسورة من الدفعة السليمة: عمود ناقص، ونص المبالغ فاضية، وعملة جديدة، وتواريخ بالمستقبل. كل وحدة بنفحصها جوا `try`، وبنطبع سبب الرفض، ولا وحدة لازم تنقبل.

</div>

> **Fail the batch, not the row.** If 40% of amounts are suddenly null, the upstream export
> broke — loading the "good" 60% quietly corrupts every downstream average. Refuse it, alert,
> and let a human look. Row-level filtering is for *expected* messiness (the negative amounts in
> `transform`); batch-level validation is for *unexpected* change.

<!-- ar -->
<div dir="rtl" lang="ar">

> **أفشل الدفعة، وليس الصف.** إذا أصبحت ٤٠٪ من المبالغ فارغة فجأة، فتصدير المصدر تعطّل. تحميل الـ ٦٠٪ "الجيدة" يفسد بهدوء كل المتوسطات بعدها. ارفضها، وأطلق تنبيهاً، ودع إنساناً ينظر. تصفية الصفوف للفوضى *المتوقعة* (مثل المبالغ السالبة في `transform`)، والتحقق على مستوى الدفعة للتغيير *غير المتوقع*.

</div>

## Step 4.1 · Put the gate in the DAG

The checks are worth nothing sitting in a file. They belong **between extract and transform**, so
a batch that fails one never reaches the warehouse.

Adding a stage to a TaskFlow DAG is one new function and one changed line. The failing task turns
red, the tasks after it never start, and Airflow tells you which one and why.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٤٫١ · ضع البوابة داخل الـ DAG**

الفحوصات لا تساوي شيئاً وهي جالسة في ملف. مكانها **بين الاستخراج والتحويل**، حتى لا تصل أي دفعة راسبة إلى المستودع.

إضافة مرحلة إلى DAG بأسلوب TaskFlow تعني دالة جديدة وسطراً واحداً معدّلاً. المهمة الفاشلة تصير حمراء، والمهام التي بعدها لا تبدأ أصلاً، و Airflow يخبرك أيها فشل ولماذا.

</div>


In [19]:
%%writefile dags/etl_pipeline.py
"""The same three functions -- now with Airflow deciding when they run."""
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parent.parent))

import pendulum
from airflow.sdk import dag, task

import etl
import validate

START = pendulum.datetime(2026, 3, 1, tz="UTC")


@dag(schedule="@daily", start_date=START, catchup=False,
     default_args={"retries": 2, "retry_delay": pendulum.duration(seconds=2)})
def etl_pipeline():
    """One run per day. Airflow picks the day; no task ever calls now()."""

    @task
    def extract(**context):
        day = (context["logical_date"] - START).days
        return str(etl.HERE / "landing" / f"events_day{day}.csv")

    @task
    def check(path):
        """The gate. Raise here and nothing downstream runs."""
        import pandas as pd
        stats = validate.validate(pd.read_csv(path, parse_dates=["occurred_at"]))
        print(f"contract ok: {stats}")
        return path

    @task
    def transform(path):
        clean = etl.transform(etl.extract(path))
        out = etl.HERE / "staging" / pathlib.Path(path).name
        out.parent.mkdir(exist_ok=True)
        clean.to_csv(out, index=False)
        print(f"{len(clean)} clean rows -> {out.name}")
        return str(out)

    @task
    def load(path):
        import pandas as pd
        n = etl.upsert(pd.read_csv(path, parse_dates=["occurred_at"]))
        print(f"offered {n} rows to the warehouse")
        return n

    load(transform(check(extract())))         # one new link in the chain


etl_pipeline()

Overwriting dags/etl_pipeline.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · الـ DAG مع بوابة التحقق**

بنعيد كتابة ملف الـ DAG وفيه مهمة جديدة اسمها `check` بتستدعي `validate.validate` على البيانات الخام بعد الاستخراج مباشرة. لو رمت استثناء، المهمة بتفشل و `transform` و `load` ما بيشتغلوا أصلاً. والتغيير الوحيد بالرسم هو السطر الأخير: صار `load(transform(check(extract())))`.

</div>


In [20]:
!airflow dags test etl_pipeline 2026-03-03 2>&1 | grep -E "contract ok|clean rows|offered"
print(f"\ntable total: {table_count()}   <- the gate passed, and the retry still changed nothing")

contract ok: {'rows': 532, 'duplicate_ids': 12, 'null_amount_rate': 0.032, 'unknown_currencies': [], 'future_rows': 0}
494 clean rows -> events_day2.csv
offered 494 rows to the warehouse



table total: 1415   <- the gate passed, and the retry still changed nothing


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تشغيل الخط مع البوابة**

بنشغّل نفس اليوم مرة تانية بعد ما ضفنا البوابة. بنشوف سطر `contract ok` ومعه الإحصاءات، وبعدها التحويل والتحميل. وعدد صفوف الجدول بيضل نفسه، لأن الخط ما زال idempotent.

</div>


---
# Part 5 — Retries and history, for free   ·   deck slides 21–22

It will fail. The question is what happens next, and what you know in the morning.

Three things decide that. The first stays yours — a transaction is a guarantee the *database*
makes, and no scheduler can give it to you. The other two used to be about sixty hand-written
lines: a retry loop with backoff and jitter, and a run-log table written at the end of every job.
Airflow gives you both as configuration.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الخامس: إعادة المحاولة والسجل، بلا كلفة**

سيفشل حتماً. السؤال هو ماذا يحدث بعد ذلك، وماذا ستعرف في الصباح.

ثلاثة أشياء تقرر ذلك. الأول يبقى مسؤوليتك، فالمعاملة ضمانة تقدّمها *قاعدة البيانات* ولا يستطيع أي منسّق أن يعطيك إياها. أما الاثنان الآخران فكانا نحو ستين سطراً مكتوبة بخط اليد: حلقة إعادة محاولة بتباعد متزايد وعشوائية، وجدول سجل تشغيل يُكتب في نهاية كل مهمة. و Airflow يعطيك الاثنين كإعدادات.

</div>


## Step 5.1 · Transactions: all of it, or none of it

This one is not Airflow's to give. `engine.begin()` commits on success and rolls back on any
exception, so a load that dies half way through leaves the table exactly as it found it.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٥٫١ · المعاملات: كل شيء أو لا شيء**

هذه ليست من عند Airflow. `engine.begin()` يثبّت عند النجاح ويتراجع عند أي استثناء، فالتحميل الذي يموت في منتصفه يترك الجدول كما وجده تماماً.

</div>


In [21]:
from sqlalchemy.dialects.sqlite import insert as sqlite_insert

before = table_count()
try:
    with engine.begin() as conn:            # begin() = commit on success, rollback on error
        conn.execute(sqlite_insert(etl.events_v2).values(
            {"event_id": 999_001, "customer_id": 1, "amount": 10.0, "currency": "USD",
             "amount_eur": 9.2, "occurred_at": pd.Timestamp("2026-03-05"),
             "occurred_date": "2026-03-05", "loaded_at": pd.Timestamp.utcnow().tz_localize(None)}))
        raise RuntimeError("something exploded half way through the load")
except RuntimeError as e:
    print("caught:", e)

after = table_count()
print(f"rows before {before}, after the failed load {after}")
assert before == after, "a failed transaction must leave nothing behind"
print("Nothing was left behind -- no half-finished batch to clean up by hand.")

caught: something exploded half way through the load
rows before 1415, after the failed load 1415
Nothing was left behind -- no half-finished batch to clean up by hand.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · المعاملات: كل شي أو لا شي**

بنعدّ الصفوف، وبعدين جوا `engine.begin()` بنضيف صف وبنرفع خطأ عمداً بنص التحميل. لأنه المعاملة فشلت، كل شي بيرجع لورا. بنعدّ من جديد، والـ `assert` بتتأكد إنه ما ضل أي صف. هاي الضمانة من قاعدة البيانات، مش من Airflow.

</div>


## Step 5.2 · Retries you declare instead of write

`retries=2` on the task. That is the whole feature. Here is a task that fails on its first
attempt on purpose, so you can watch Airflow deal with it.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٥٫٢ · إعادة محاولة تُعلَن بدل أن تُكتب**

`retries=2` على المهمة. هذه هي الميزة كلها. وهذه مهمة تفشل في محاولتها الأولى عن قصد، لتشاهد كيف يتصرف Airflow.

</div>


In [22]:
%%writefile dags/flaky.py
"""Retries you declare instead of write. This task fails once, on purpose."""
import pendulum
from airflow.sdk import dag, task


@dag(schedule=None, start_date=pendulum.datetime(2026, 3, 1, tz="UTC"), catchup=False)
def flaky_pipeline():

    @task(retries=2, retry_delay=pendulum.duration(seconds=1))
    def call_a_flaky_api(**context):
        attempt = context["ti"].try_number
        print(f"attempt {attempt} ...")
        if attempt == 1:
            raise ConnectionError("upstream API timed out")      # the transient kind
        print(f"succeeded on attempt {attempt}")
        return attempt

    call_a_flaky_api()


flaky_pipeline()

Writing dags/flaky.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مهمة تفشل مرة عن قصد**

DAG صغير فيه مهمة وحدة، و `retries=2` مع `retry_delay` ثانية وحدة. `context["ti"].try_number` بترجع رقم المحاولة الحالية، فبنرمي `ConnectionError` بس بالمحاولة الأولى. هيك بنشوف Airflow وهو بيعيد المحاولة لحاله، بدون ما نكتب ولا حلقة.

</div>


In [23]:
!airflow dags test flaky_pipeline 2026-03-01 2>&1 | grep -E "attempt|ConnectionError|succeeded"

attempt 1 ...
    raise ConnectionError("upstream API timed out")      # the transient kind
ConnectionError: upstream API timed out


attempt 2 ...
succeeded on attempt 2


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مشاهدة إعادة المحاولة**

بنشغّل الـ DAG وبنفلتر الإخراج. بنشوف المحاولة الأولى بتفشل بـ `ConnectionError`، وبعدين المحاولة التانية بتنجح. كل هاد من سطر إعدادات واحد.

</div>


**Retry transient failures only.** A timeout, a closed connection, a rate limit — those are worth
retrying. A `DataContractError` from Part 4 is not: the data is wrong and it will still be wrong
in two seconds. Retrying it just delays the alert.

<!-- ar -->
<div dir="rtl" lang="ar">

**أعد المحاولة مع الأعطال العابرة فقط.** انتهاء مهلة، أو اتصال مقطوع، أو تجاوز حد الطلبات: هذه تستحق إعادة المحاولة. أما `DataContractError` من الجزء الرابع فلا: البيانات خاطئة وستبقى خاطئة بعد ثانيتين، وإعادة المحاولة تؤخّر التنبيه لا أكثر.

</div>


## Step 5.3 · The history you did not have to write

Someone asks at 8am why yesterday's numbers look wrong. Every run, every task, every attempt is
already recorded — you did not add a single line for it.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٥٫٣ · السجل الذي لم تكتبه**

يسألك أحدهم الساعة الثامنة صباحاً: لماذا أرقام الأمس تبدو خاطئة؟ كل تشغيل، وكل مهمة، وكل محاولة مسجّلة أصلاً، ولم تضف سطراً واحداً من أجل ذلك.

</div>


In [24]:
print("=== every run of this DAG " + "=" * 46)
!airflow dags list-runs etl_pipeline 2>&1 | head -7

run_id = !airflow dags list-runs etl_pipeline -o plain 2>/dev/null | awk 'NR==2{print $2}'
print("\n=== every task inside the latest run " + "=" * 33)
!airflow tasks states-for-dag-run etl_pipeline {run_id[0]} 2>&1 | head -7

=== every run of this DAG ==============================================


dag_id       | run_id                                   | state   | run_after                        | logical_date              | start_date                | end_date                        
=============+==========================================+=========+==================================+===========================+===========================+=================================
etl_pipeline | manual__2026-09-20T12:41:34.271550+00:00 | success | 2026-09-20T12:41:34.271550+00:00 | 2026-03-03T00:00:00+00:00 | 2026-03-03T00:00:00+00:00 | 2026-09-20T12:41:37.734271+00:00
etl_pipeline | manual__2026-09-20T12:41:08.634379+00:00 | success | 2026-09-20T12:41:08.634379+00:00 | 2026-03-02T00:00:00+00:00 | 2026-03-02T00:00:00+00:00 | 2026-09-20T12:41:11.956873+00:00
etl_pipeline | manual__2026-09-20T12:41:00.292915+00:00 | success | 2026-09-20T12:41:00.292915+00:00 | 2026-03-01T00:00:00+00:00 | 2026-03-01T00:00:00+00:00 | 2026-09-20T12:41:03.911437+00:00
                                        


=== every task inside the latest run =================================


dag_id       | logical_date              | task_id   | state   | start_date                       | end_date                        
=============+===========================+===========+=========+==================================+=================================
etl_pipeline | 2026-03-03T00:00:00+00:00 | extract   | success | 2026-09-20T12:41:35.148985+00:00 | 2026-09-20T12:41:37.441618+00:00
etl_pipeline | 2026-03-03T00:00:00+00:00 | check     | success | 2026-09-20T12:41:37.484606+00:00 | 2026-09-20T12:41:37.534182+00:00
etl_pipeline | 2026-03-03T00:00:00+00:00 | transform | success | 2026-09-20T12:41:37.566152+00:00 | 2026-09-20T12:41:37.616145+00:00
etl_pipeline | 2026-03-03T00:00:00+00:00 | load      | success | 2026-09-20T12:41:37.644113+00:00 | 2026-09-20T12:41:37.716415+00:00
                                                                                                                                    


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · سجل التشغيلات وحالة كل مهمة**

`airflow dags list-runs` بتعرض كل تشغيلات الـ DAG: الحالة، والتاريخ المنطقي، ووقت البداية والنهاية. وبعدين بناخد `run_id` لآخر تشغيل بصيغة `plain` وبنمرره لـ `states-for-dag-run` عشان نشوف حالة كل مهمة لحالها: `extract` و `check` و `transform` و `load`. هاد كله بديل جدول `etl_runs` يلي كنا نكتبه بإيدنا، وبدون ولا سطر كود: لما يفشل شي، هون بتشوف أي مهمة بالضبط وقفت.

</div>


## What you did not have to write

Every row here was working code in the hand-rolled version of this session. All of it is gone,
and the pipeline does more than it did before.

| You used to write | Airflow gives you | Lines saved |
|---|---|---|
| an `etl_watermark` table, `get_watermark`, `set_watermark` | `context["logical_date"]` | ~30 |
| a `lateness` window to catch late rows | re-run any day; the upsert absorbs it | ~5 |
| `with_retry`, exponential backoff, jitter | `retries=2, retry_delay=...` | ~23 |
| an `etl_runs` table and a `run_job` wrapper | `dags list-runs`, `states-for-dag-run` | ~35 |
| two matplotlib charts to read that table | the same commands, plus a UI you did not start | ~12 |

What did **not** go away: the pure `transform`, the primary key and upsert, and the validation
contract. That is the line. **The orchestrator runs the job; it is not the job** — and everything
on the left of this table was orchestration you were writing by hand.

<!-- ar -->
<div dir="rtl" lang="ar">

**ما لم تعد مضطراً لكتابته**

كل سطر في هذا الجدول كان كوداً عاملاً في النسخة اليدوية من هذه الجلسة. اختفى كله، والخط يفعل أكثر مما كان يفعل.

| كنت تكتب | يعطيك Airflow | أسطر موفّرة |
|---|---|---|
| جدول `etl_watermark` ودالتَي القراءة والكتابة | `context["logical_date"]` | ~٣٠ |
| نافذة تأخير لالتقاط الصفوف المتأخرة | أعد تشغيل أي يوم، والـ upsert يستوعبه | ~٥ |
| `with_retry` وتباعد متزايد وعشوائية | `retries=2, retry_delay=...` | ~٢٣ |
| جدول `etl_runs` وغلاف `run_job` | `dags list-runs` و `states-for-dag-run` | ~٣٥ |
| رسمان بيانيان لقراءة ذلك الجدول | الأوامر نفسها، ومعها واجهة لم تشغّلها | ~١٢ |

وما **لم** يختفِ: دالة `transform` النقية، والمفتاح الأساسي مع الـ upsert، وعقد التحقق. هنا يقع الخط الفاصل. **المنسّق يشغّل المهمة، وهو ليس المهمة**، وكل ما في يمين هذا الجدول كان تنسيقاً تكتبه بيدك.

</div>


---
# Reference — worth knowing, not demonstrated here

| Topic | One-line version |
|---|---|
| Postgres instead of SQLite | change one URL: `postgresql+psycopg2://user:pw@host/db`; use `postgresql.insert` for upsert |
| ELT vs ETL | load raw first, transform in the warehouse — sensible when the warehouse is the compute |
| The Airflow UI | `airflow standalone` starts a scheduler and a web server on :8080 — the graph, the logs, the retry button |
| Backfill | `airflow backfill create` re-runs a date range; idempotency is what makes that safe |
| `catchup=True` | on a real schedule, Airflow runs every day it missed. Off here so the notebook stays predictable |
| Sensors and assets | wait for a file, or trigger when another DAG's output lands, instead of a fixed time |
| Executors | `dags test` runs in one process; real deployments use Celery or Kubernetes to run tasks in parallel |
| Chunked loads | `pd.read_csv(chunksize=...)` and `executemany` when a file will not fit in memory |
| Change data capture | Debezium and friends, when a daily batch is not fresh enough |
| Great Expectations | a full validation framework once these hand-written checks grow |

## Recap

| You wanted to… | Do this |
|---|---|
| test the business logic fast | keep `transform` a pure function |
| survive being run twice | primary key + `ON CONFLICT DO UPDATE` |
| stop reprocessing everything | take the window from `logical_date`, not the clock |
| not lose late-arriving rows | re-run the day; the upsert makes it free |
| refuse a broken batch | validate at the boundary, raise, do not partially load |
| avoid half-finished loads | one transaction per batch (`engine.begin()`) |
| survive a flaky source | `retries=2` — transient errors only, never a contract error |
| answer "what happened last night" | `airflow dags list-runs`, or the UI |

## What to do at work tomorrow

1. Run your pipeline twice on the same input. If the table changes, **fix that first** — nothing
   else matters until it is idempotent.
2. Put a primary key on the natural key and switch the load to an upsert.
3. Find every `datetime.now()` in a scheduled job and replace it with the run's logical date.
4. Write five validation checks. Row count, null rate, allowed values, no future dates, schema.
5. Delete your retry loop and your run-log table. Your orchestrator already has both.

<!-- ar -->
<div dir="rtl" lang="ar">

**مرجع: يستحق المعرفة، ولم نجرّبه هنا**

| الموضوع | بسطر واحد |
|---|---|
| Postgres بدل SQLite | غيّر رابطاً واحداً: `postgresql+psycopg2://user:pw@host/db`، واستخدم `postgresql.insert` للـ upsert |
| ELT مقابل ETL | حمّل البيانات الخام أولاً، وحوّلها داخل مستودع البيانات، وهذا منطقي عندما يكون المستودع هو مكان الحساب |
| واجهة Airflow | `airflow standalone` يشغّل مجدولاً وخادم ويب على المنفذ 8080: الرسم، والسجلات، وزر إعادة المحاولة |
| إعادة معالجة القديم | `airflow backfill create` يعيد تشغيل مدى من التواريخ، وعدم التكرار هو ما يجعل ذلك آمناً |
| `catchup=True` | على جدول حقيقي، يشغّل Airflow كل يوم فاته. أطفأناه هنا ليبقى الدفتر متوقَّعاً |
| المستشعرات والأصول | انتظر وصول ملف، أو انطلق حين تجهز مخرجات DAG آخر، بدل وقت ثابت |
| المنفّذات | `dags test` يعمل في عملية واحدة، والأنظمة الحقيقية تستخدم Celery أو Kubernetes لتشغيل المهام بالتوازي |
| التحميل على أجزاء | `pd.read_csv(chunksize=...)` و `executemany` عندما لا يتسع الملف في الذاكرة |
| التقاط التغييرات (CDC) | Debezium وأمثاله، عندما لا تكفي دفعة يومية |
| Great Expectations | إطار تحقق كامل عندما تكبر هذه الفحوصات اليدوية |

**الخلاصة**

| تريد أن… | افعل هذا |
|---|---|
| تختبر منطق العمل بسرعة | اجعل `transform` دالة نقية |
| تصمد أمام التشغيل مرتين | مفتاح أساسي مع `ON CONFLICT DO UPDATE` |
| تتوقف عن إعادة معالجة كل شيء | خذ النافذة من `logical_date` لا من الساعة |
| لا تخسر الصفوف المتأخرة | أعد تشغيل اليوم، والـ upsert يجعلها بلا كلفة |
| ترفض دفعة معطوبة | تحقّق عند الحدود، وارمِ استثناءً، ولا تحمّل جزئياً |
| تتجنب التحميل نصف المكتمل | معاملة واحدة لكل دفعة (`engine.begin()`) |
| تصمد أمام مصدر متقطّع | `retries=2` للأعطال العابرة فقط، وليس لخطأ العقد أبداً |
| تجيب عن "ماذا جرى ليلة أمس" | `airflow dags list-runs` أو الواجهة |

**ماذا تفعل في العمل غداً**

1. شغّل خطك مرتين على نفس المدخل. إذا تغيّر الجدول، **أصلح ذلك أولاً**، فلا شيء آخر يهم قبل أن يصير idempotent.
2. ضع مفتاحاً أساسياً على المفتاح الطبيعي، وحوّل التحميل إلى upsert.
3. ابحث عن كل `datetime.now()` في مهمة مجدولة واستبدلها بالتاريخ المنطقي للتشغيل.
4. اكتب خمسة فحوص تحقق: عدد الصفوف، ونسبة الفراغات، والقيم المسموحة، ولا تواريخ مستقبلية، والمخطط.
5. احذف حلقة إعادة المحاولة وجدول سجل التشغيل. منسّقك يملك الاثنين أصلاً.

</div>


## Cleanup

Everything above happened in the sandbox. This removes it.

<!-- ar -->
<div dir="rtl" lang="ar">

**التنظيف**

كل ما سبق حدث داخل مجلد التجربة، وهذه الخلية تحذفه.

</div>

In [25]:
import shutil, os
os.chdir(BASE)
shutil.rmtree(PROJ, ignore_errors=True)
print("sandbox removed")

sandbox removed


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · التنظيف**

بنرجع لمجلد العمل الأصلي `BASE`، وبنمسح مجلد `etl_demo` بالكامل، يعني نرجع الوضع متل ما كان قبل الدفتر.

</div>